# 02 - Run Benchmarks

Run each retrieval strategy against every generated question, measure latency, and evaluate answers with RAGAS. Intermediate CSVs are saved so the run can resume after interruptions.


## Cell 1 - Setup and load dataset

Load the test dataset, configure the strategy list, and prepare output directories.


In [ ]:

import os, json, time, signal, math, uuid, requests
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

API_BASE = os.getenv("BENCHMARK_API_BASE", "http://localhost:8000").rstrip("/")
API_KEY = os.getenv("BENCHMARK_API_KEY", "your-api-key-here")
PROJECT_ID = os.getenv("BENCHMARK_PROJECT_ID", "your-project-id-here")
DATASET_FILE = os.getenv("BENCHMARK_DATASET_FILE", "results/test_dataset.json")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

STRATEGIES = ["vanilla", "hybrid", "rerank", "hyde"]
N_RUNS = int(os.getenv("BENCHMARK_N_RUNS", "1"))  # set to 3 for final benchmarks, 1 for quick testing
TOP_K = int(os.getenv("BENCHMARK_TOP_K", "5"))
REQUEST_TIMEOUT_SECONDS = int(os.getenv("BENCHMARK_REQUEST_TIMEOUT_SECONDS", "120"))
RAGAS_TIMEOUT_SECONDS = int(os.getenv("BENCHMARK_RAGAS_TIMEOUT_SECONDS", "180"))
SLEEP_SECONDS = float(os.getenv("BENCHMARK_SLEEP_SECONDS", "1"))
RESUME = os.getenv("BENCHMARK_RESUME", "true").lower() == "true"

RAGAS_METRICS = [faithfulness, answer_relevancy, context_precision, context_recall]
METRIC_NAMES = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

if API_KEY == "your-api-key-here" or PROJECT_ID == "your-project-id-here":
    raise ValueError("Set BENCHMARK_API_KEY and BENCHMARK_PROJECT_ID before running benchmarks.")

with open(DATASET_FILE, "r", encoding="utf-8") as file:
    records = json.load(file)

test_df = pd.DataFrame(records)
if test_df.empty:
    raise RuntimeError(f"Dataset is empty: {DATASET_FILE}")
if "id" not in test_df.columns:
    test_df["id"] = [f"q_{i + 1:04d}" for i in range(len(test_df))]

session = requests.Session()
session.headers.update({"X-API-Key": API_KEY, "Content-Type": "application/json"})

print(f"Loaded {len(test_df)} benchmark questions")
display(test_df[["id", "difficulty", "question", "answer"]].head())


## Cell 2 - Helper: query and evaluate one strategy

Call `/api/query`, record end-to-end latency, run RAGAS, and return a single result row. API errors and RAGAS failures are captured as row-level errors instead of stopping the whole benchmark.


In [ ]:

class RagasTimeoutError(TimeoutError):
    pass


def run_with_timeout(func, timeout_seconds):
    if os.name == "nt":
        return func()

    def handler(signum, frame):
        raise RagasTimeoutError(f"RAGAS evaluation exceeded {timeout_seconds}s")

    previous_handler = signal.signal(signal.SIGALRM, handler)
    signal.alarm(timeout_seconds)
    try:
        return func()
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, previous_handler)


def evaluate_ragas_safely(dataset):
    def _evaluate():
        try:
            return evaluate(dataset, metrics=RAGAS_METRICS, raise_exceptions=False)
        except TypeError:
            return evaluate(dataset, metrics=RAGAS_METRICS)

    scores = {name: np.nan for name in METRIC_NAMES}
    error = None
    try:
        result = run_with_timeout(_evaluate, RAGAS_TIMEOUT_SECONDS)
        if hasattr(result, "to_pandas"):
            row = result.to_pandas().iloc[0].to_dict()
        else:
            row = dict(result)
        for name in METRIC_NAMES:
            value = row.get(name, np.nan)
            if isinstance(value, (list, tuple, np.ndarray, pd.Series)):
                value = value[0] if len(value) else np.nan
            scores[name] = float(value) if pd.notna(value) else np.nan
    except Exception as exc:
        error = str(exc)
    return scores, error


def run_single_query(question, ground_truth, strategy, project_id, api_key):
    """
    1. Call POST /api/query with the question and strategy
    2. Record total latency
    3. Run RAGAS evaluation on the response
    4. Return all metrics as a dict
    """
    payload = {
        "text": question,
        "project_id": project_id,
        "strategy": strategy,
        "top_k": TOP_K,
        "session_id": f"benchmark-{strategy}-{uuid.uuid4()}",
    }

    started_at = time.time()
    response_payload = {}
    api_error = None
    status_code = None

    try:
        response = session.post(
            f"{API_BASE}/api/query",
            json=payload,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        status_code = response.status_code
        latency = (time.time() - started_at) * 1000
        if not response.ok:
            try:
                detail = response.json().get("detail", response.text)
            except Exception:
                detail = response.text
            raise RuntimeError(f"HTTP {response.status_code}: {detail}")
        response_payload = response.json()
    except Exception as exc:
        latency = (time.time() - started_at) * 1000
        api_error = str(exc)

    answer = response_payload.get("answer", "")
    sources = response_payload.get("sources", []) or []
    contexts = [str(c.get("text", "")) for c in sources if c.get("text")]

    ragas_scores = {name: np.nan for name in METRIC_NAMES}
    ragas_error = None
    if answer and contexts and api_error is None:
        dataset = Dataset.from_dict(
            {
                "question": [question],
                "answer": [answer],
                "contexts": [contexts],
                "ground_truth": [ground_truth],
            }
        )
        ragas_scores, ragas_error = evaluate_ragas_safely(dataset)
    elif api_error is None:
        ragas_error = "Skipped RAGAS because answer or contexts were empty"

    return {
        "question": question,
        "ground_truth": ground_truth,
        "answer": answer,
        "strategy": strategy,
        "executed_strategy": response_payload.get("retrieval_strategy"),
        "routing_decision": response_payload.get("routing_decision"),
        "latency_ms": latency,
        "api_reported_latency_ms": response_payload.get("total_latency_ms"),
        "n_sources": len(sources),
        "status_code": status_code,
        "api_error": api_error,
        "ragas_error": ragas_error,
        "estimated_cost_usd": response_payload.get("estimated_cost_usd"),
        **ragas_scores,
    }


## Cell 3 - Run benchmarks

Loop through every strategy, question, and run. Per-strategy CSV files are updated every 10 rows and at the end.


In [ ]:

def existing_results_for(strategy):
    path = RESULTS_DIR / f"{strategy}_results.csv"
    if RESUME and path.exists():
        existing = pd.read_csv(path)
        completed = set(zip(existing.get("question_id", []), existing.get("run", [])))
        return existing.to_dict("records"), completed
    return [], set()


all_results = []

for strategy in STRATEGIES:
    strategy_results, completed = existing_results_for(strategy)
    output_path = RESULTS_DIR / f"{strategy}_results.csv"
    total_iterations = len(test_df) * N_RUNS

    with tqdm(total=total_iterations, desc=f"Strategy: {strategy}") as progress:
        progress.update(len(completed))
        for _, row in test_df.iterrows():
            question_id = row["id"]
            for run_idx in range(1, N_RUNS + 1):
                key = (question_id, run_idx)
                if key in completed:
                    continue

                result = run_single_query(
                    question=row["question"],
                    ground_truth=row["answer"],
                    strategy=strategy,
                    project_id=PROJECT_ID,
                    api_key=API_KEY,
                )
                result.update(
                    {
                        "question_id": question_id,
                        "run": run_idx,
                        "difficulty": row.get("difficulty"),
                        "document_id": row.get("document_id"),
                        "document_name": row.get("document_name"),
                        "chunk_id": row.get("chunk_id"),
                    }
                )
                strategy_results.append(result)
                completed.add(key)

                if len(strategy_results) % 10 == 0:
                    pd.DataFrame(strategy_results).to_csv(output_path, index=False)

                progress.update(1)
                time.sleep(SLEEP_SECONDS)

    strategy_df = pd.DataFrame(strategy_results)
    strategy_df.to_csv(output_path, index=False)
    all_results.extend(strategy_results)
    print(f"Saved {len(strategy_df)} rows to {output_path}")

all_results_df = pd.DataFrame(all_results)
all_results_df.to_csv(RESULTS_DIR / "all_results.csv", index=False)
print(f"Benchmark complete. Combined rows: {len(all_results_df)}")


## Cell 4 - Summary statistics

Compute means, standard deviations, and latency percentiles for each strategy.


In [ ]:

result_frames = []
for strategy in STRATEGIES:
    path = RESULTS_DIR / f"{strategy}_results.csv"
    if path.exists():
        result_frames.append(pd.read_csv(path))

if not result_frames:
    raise RuntimeError("No result files found. Run Cell 3 first.")

results_df = pd.concat(result_frames, ignore_index=True)
for column in METRIC_NAMES + ["latency_ms", "api_reported_latency_ms", "estimated_cost_usd"]:
    if column in results_df.columns:
        results_df[column] = pd.to_numeric(results_df[column], errors="coerce")

summary_rows = []
for strategy, group in results_df.groupby("strategy"):
    row = {"strategy": strategy, "n": len(group)}
    for metric in METRIC_NAMES:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_std"] = group[metric].std()
    row["latency_mean_ms"] = group["latency_ms"].mean()
    row["latency_std_ms"] = group["latency_ms"].std()
    row["latency_p50_ms"] = group["latency_ms"].quantile(0.50)
    row["latency_p95_ms"] = group["latency_ms"].quantile(0.95)
    row["latency_p99_ms"] = group["latency_ms"].quantile(0.99)
    row["api_errors"] = group["api_error"].notna().sum() if "api_error" in group else 0
    row["ragas_errors"] = group["ragas_error"].notna().sum() if "ragas_error" in group else 0
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).sort_values("faithfulness_mean", ascending=False)
display(
    summary.style.format(
        {
            "faithfulness_mean": "{:.3f}",
            "faithfulness_std": "{:.3f}",
            "answer_relevancy_mean": "{:.3f}",
            "answer_relevancy_std": "{:.3f}",
            "context_precision_mean": "{:.3f}",
            "context_precision_std": "{:.3f}",
            "context_recall_mean": "{:.3f}",
            "context_recall_std": "{:.3f}",
            "latency_mean_ms": "{:.0f}",
            "latency_std_ms": "{:.0f}",
            "latency_p50_ms": "{:.0f}",
            "latency_p95_ms": "{:.0f}",
            "latency_p99_ms": "{:.0f}",
        }
    )
)
summary.to_csv(RESULTS_DIR / "summary_statistics.csv", index=False)
